## What is Molecular Standardization?

Molecular standardization is the process of converting chemical structures into a consistent and uniform representation.

It removes unnecessary differences in molecular representations while preserving the chemical identity of the compound.

Standardized molecules improve data quality, eliminate inconsistencies, and ensure reliable cheminformatics analyses.

## Intuition: Standardizing Home Addresses

Imagine three people write the same address differently:

- 221B Baker Street
- 221-B Baker St.
- Baker Street 221B

A human immediately understands that all three refer to the same location.

A computer, however, may treat them as different addresses.

Therefore, addresses are standardized before being stored in a database.

Molecules face the same problem.

The same chemical compound may be represented in multiple valid ways.

Molecular standardization converts these different representations into a consistent format.

## What Exactly Do We Standardize?

Molecular standardization is not a single operation.

Instead, it consists of several preprocessing steps, each designed to remove a specific type of inconsistency from molecular data.

Depending on the source of the data and the intended application, one or more of these steps may be applied before further analysis.

### 1. Salt Removal

Many drug molecules are stored as pharmaceutical salts to improve stability or solubility.

Examples include:

- Hydrochloride salts
- Sodium salts
- Potassium salts

For cheminformatics analyses, only the parent molecule is usually required.

Therefore, counter ions are removed during molecular standardization.

### 2. Charge Neutralization

Some molecules are represented in charged forms.

While chemically correct, charged structures may lead to inconsistent descriptor and fingerprint calculations.

Charge neutralization converts molecules into their neutral form whenever appropriate.

### 3. Canonical SMILES

The same molecule can often be represented by multiple valid SMILES strings.

Canonical SMILES converts these different representations into one unique, standardized SMILES.

This ensures that identical molecules always have the same textual representation.

### 4. Tautomer Standardization

Many molecules exist in multiple tautomeric forms due to the movement of hydrogen atoms and double bonds.

Although these forms represent the same chemical compound, computers may treat them as different molecules.

Tautomer standardization converts all tautomeric forms into a single, consistent representation.|

### 5. Duplicate Detection

Large molecular databases often contain duplicate compounds represented by different SMILES strings.

The InChIKey provides a standardized identifier that enables rapid identification and removal of duplicate molecules.

# Example
Suppose we receive a small molecular dataset from different chemical databases.

The dataset contains:

- Pharmaceutical salts
- Charged molecules
- Multiple SMILES representations of the same compound
- Tautomeric forms

Our objective is to standardize these molecules before performing cheminformatics analyses.

In [2]:
import pandas as pd

compounds = {
    "Ethanol": "CCO",

    "Acetic Acid": "CC(=O)O",

    "Sodium Acetate": "CC(=O)[O-].[Na+]",

    "Metformin Hydrochloride": "CN(C)C(=N)N.Cl",

    "Ammonium": "[NH4+]",

    "Acetylacetone (Keto)": "CC(=O)CC(=O)C",

    "Acetylacetone (Enol)": "CC(=O)C=C(O)C",

    "Paracetamol": "CC(=O)NC1=CC=C(C=C1)O",

    "Paracetamol (Alternative SMILES)": "OC1=CC=C(NC(C)=O)C=C1",

    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O"
}

df = pd.DataFrame(
    compounds.items(),
    columns=["Compound", "SMILES"]
)

df

,Compound,SMILES
0,Ethanol,CCO
1,Acetic Acid,CC(=O)O
2,Sodium Acetate,CC(=O)[O-].[Na+]
3,Metformin Hydrochloride,CN(C)C(=N)N.Cl
4,Ammonium,[NH4+]
5,Acetylacetone (Keto),CC(=O)CC(=O)C
6,Acetylacetone (Enol),CC(=O)C=C(O)C
7,Paracetamol,CC(=O)NC1=CC=C(C=C1)O
8,Paracetamol (Alternative SMILES),OC1=CC=C(NC(C)=O)C=C1
9,Aspirin,CC(=O)Oc1ccccc1C(=O)O


In [3]:
# Salt Removal

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

# Salt remover
remover = rdMolStandardize.FragmentRemover()

salt_removed = []

for smiles in df["SMILES"]:

    mol = Chem.MolFromSmiles(smiles)

    if mol:
        mol = remover.remove(mol)
        salt_removed.append(Chem.MolToSmiles(mol))
    else:
        salt_removed.append(None)

df["Salt_Removed"] = salt_removed

df

[15:31:38] Running FragmentRemover
[15:31:38] Running FragmentRemover
[15:31:38] Running FragmentRemover
[15:31:38] Removed fragment: sodium
[15:31:38] Running FragmentRemover
[15:31:38] Removed fragment: chlorine
[15:31:38] Running FragmentRemover
[15:31:38] Running FragmentRemover
[15:31:38] Running FragmentRemover
[15:31:38] Running FragmentRemover
[15:31:38] Running FragmentRemover
[15:31:38] Running FragmentRemover


,Compound,SMILES,Salt_Removed
0,Ethanol,CCO,CCO
1,Acetic Acid,CC(=O)O,CC(=O)O
2,Sodium Acetate,CC(=O)[O-].[Na+],CC(=O)[O-]
3,Metformin Hydrochloride,CN(C)C(=N)N.Cl,CN(C)C(=N)N
4,Ammonium,[NH4+],[NH4+]
5,Acetylacetone (Keto),CC(=O)CC(=O)C,CC(=O)CC(C)=O
6,Acetylacetone (Enol),CC(=O)C=C(O)C,CC(=O)C=C(C)O
7,Paracetamol,CC(=O)NC1=CC=C(C=C1)O,CC(=O)Nc1ccc(O)cc1
8,Paracetamol (Alternative SMILES),OC1=CC=C(NC(C)=O)C=C1,CC(=O)Nc1ccc(O)cc1
9,Aspirin,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O


In [4]:
# Charge Neutralization

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

# Create uncharger
uncharger = rdMolStandardize.Uncharger()

neutralized = []

for smiles in df["Salt_Removed"]:

    mol = Chem.MolFromSmiles(smiles)

    if mol:
        mol = uncharger.uncharge(mol)
        neutralized.append(Chem.MolToSmiles(mol))
    else:
        neutralized.append(None)

# Add new column
df["Neutralized"] = neutralized

df

[15:33:46] Running Uncharger
[15:33:46] Running Uncharger
[15:33:46] Running Uncharger
[15:33:46] Removed negative charge.
[15:33:46] Running Uncharger
[15:33:46] Running Uncharger
[15:33:46] Removed positive charge.
[15:33:46] Running Uncharger
[15:33:46] Running Uncharger
[15:33:46] Running Uncharger
[15:33:46] Running Uncharger
[15:33:46] Running Uncharger


,Compound,SMILES,Salt_Removed,Neutralized
0,Ethanol,CCO,CCO,CCO
1,Acetic Acid,CC(=O)O,CC(=O)O,CC(=O)O
2,Sodium Acetate,CC(=O)[O-].[Na+],CC(=O)[O-],CC(=O)O
3,Metformin Hydrochloride,CN(C)C(=N)N.Cl,CN(C)C(=N)N,CN(C)C(=N)N
4,Ammonium,[NH4+],[NH4+],N
5,Acetylacetone (Keto),CC(=O)CC(=O)C,CC(=O)CC(C)=O,CC(=O)CC(C)=O
6,Acetylacetone (Enol),CC(=O)C=C(O)C,CC(=O)C=C(C)O,CC(=O)C=C(C)O
7,Paracetamol,CC(=O)NC1=CC=C(C=C1)O,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1
8,Paracetamol (Alternative SMILES),OC1=CC=C(NC(C)=O)C=C1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1
9,Aspirin,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O


In [5]:
# Canonical SMILES

canonical_smiles = []

for smiles in df["Neutralized"]:

    mol = Chem.MolFromSmiles(smiles)

    if mol:
        canonical_smiles.append(Chem.MolToSmiles(mol, canonical=True))
    else:
        canonical_smiles.append(None)

df["Canonical_SMILES"] = canonical_smiles

df

,Compound,SMILES,Salt_Removed,Neutralized,Canonical_SMILES
0,Ethanol,CCO,CCO,CCO,CCO
1,Acetic Acid,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O
2,Sodium Acetate,CC(=O)[O-].[Na+],CC(=O)[O-],CC(=O)O,CC(=O)O
3,Metformin Hydrochloride,CN(C)C(=N)N.Cl,CN(C)C(=N)N,CN(C)C(=N)N,CN(C)C(=N)N
4,Ammonium,[NH4+],[NH4+],N,N
5,Acetylacetone (Keto),CC(=O)CC(=O)C,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O
6,Acetylacetone (Enol),CC(=O)C=C(O)C,CC(=O)C=C(C)O,CC(=O)C=C(C)O,CC(=O)C=C(C)O
7,Paracetamol,CC(=O)NC1=CC=C(C=C1)O,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1
8,Paracetamol (Alternative SMILES),OC1=CC=C(NC(C)=O)C=C1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1
9,Aspirin,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O


In [6]:
# Tautomer Standardization

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

# Create tautomer enumerator
enumerator = rdMolStandardize.TautomerEnumerator()

standardized = []

for smiles in df["Canonical_SMILES"]:

    mol = Chem.MolFromSmiles(smiles)

    if mol:
        mol = enumerator.Canonicalize(mol)
        standardized.append(Chem.MolToSmiles(mol))
    else:
        standardized.append(None)

df["Canonical_Tautomer"] = standardized

df

,Compound,SMILES,Salt_Removed,Neutralized,Canonical_SMILES,Canonical_Tautomer
0,Ethanol,CCO,CCO,CCO,CCO,CCO
1,Acetic Acid,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O
2,Sodium Acetate,CC(=O)[O-].[Na+],CC(=O)[O-],CC(=O)O,CC(=O)O,CC(=O)O
3,Metformin Hydrochloride,CN(C)C(=N)N.Cl,CN(C)C(=N)N,CN(C)C(=N)N,CN(C)C(=N)N,CN(C)C(=N)N
4,Ammonium,[NH4+],[NH4+],N,N,N
5,Acetylacetone (Keto),CC(=O)CC(=O)C,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O
6,Acetylacetone (Enol),CC(=O)C=C(O)C,CC(=O)C=C(C)O,CC(=O)C=C(C)O,CC(=O)C=C(C)O,CC(=O)CC(C)=O
7,Paracetamol,CC(=O)NC1=CC=C(C=C1)O,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1
8,Paracetamol (Alternative SMILES),OC1=CC=C(NC(C)=O)C=C1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1
9,Aspirin,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O


In [7]:
# Duplicate Detection

from rdkit import Chem

inchikeys = []

for smiles in df["Canonical_Tautomer"]:

    mol = Chem.MolFromSmiles(smiles)

    if mol:
        inchikey = Chem.MolToInchiKey(mol)
        inchikeys.append(inchikey)
    else:
        inchikeys.append(None)

df["InChIKey"] = inchikeys

df

,Compound,SMILES,Salt_Removed,Neutralized,Canonical_SMILES,Canonical_Tautomer,InChIKey
0,Ethanol,CCO,CCO,CCO,CCO,CCO,LFQSCWFLJHTTHZ-UHFFFAOYSA-N
1,Acetic Acid,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O,QTBSBXVTEAMEQO-UHFFFAOYSA-N
2,Sodium Acetate,CC(=O)[O-].[Na+],CC(=O)[O-],CC(=O)O,CC(=O)O,CC(=O)O,QTBSBXVTEAMEQO-UHFFFAOYSA-N
3,Metformin Hydrochloride,CN(C)C(=N)N.Cl,CN(C)C(=N)N,CN(C)C(=N)N,CN(C)C(=N)N,CN(C)C(=N)N,SWSQBOPZIKWTGO-UHFFFAOYSA-N
4,Ammonium,[NH4+],[NH4+],N,N,N,QGZKDVFQNNGYKY-UHFFFAOYSA-N
5,Acetylacetone (Keto),CC(=O)CC(=O)C,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O,YRKCREAYFQTBPV-UHFFFAOYSA-N
6,Acetylacetone (Enol),CC(=O)C=C(O)C,CC(=O)C=C(C)O,CC(=O)C=C(C)O,CC(=O)C=C(C)O,CC(=O)CC(C)=O,YRKCREAYFQTBPV-UHFFFAOYSA-N
7,Paracetamol,CC(=O)NC1=CC=C(C=C1)O,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,RZVAJINKPMORJF-UHFFFAOYSA-N
8,Paracetamol (Alternative SMILES),OC1=CC=C(NC(C)=O)C=C1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,RZVAJINKPMORJF-UHFFFAOYSA-N
9,Aspirin,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,BSYNRYMUTXBXSQ-UHFFFAOYSA-N


In [9]:
# find Duplicate Molecules
duplicates = df[df.duplicated(subset="InChIKey", keep=False)]

duplicates

,Compound,SMILES,Salt_Removed,Neutralized,Canonical_SMILES,Canonical_Tautomer,InChIKey
1,Acetic Acid,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O,QTBSBXVTEAMEQO-UHFFFAOYSA-N
2,Sodium Acetate,CC(=O)[O-].[Na+],CC(=O)[O-],CC(=O)O,CC(=O)O,CC(=O)O,QTBSBXVTEAMEQO-UHFFFAOYSA-N
5,Acetylacetone (Keto),CC(=O)CC(=O)C,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O,YRKCREAYFQTBPV-UHFFFAOYSA-N
6,Acetylacetone (Enol),CC(=O)C=C(O)C,CC(=O)C=C(C)O,CC(=O)C=C(C)O,CC(=O)C=C(C)O,CC(=O)CC(C)=O,YRKCREAYFQTBPV-UHFFFAOYSA-N
7,Paracetamol,CC(=O)NC1=CC=C(C=C1)O,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,RZVAJINKPMORJF-UHFFFAOYSA-N
8,Paracetamol (Alternative SMILES),OC1=CC=C(NC(C)=O)C=C1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,RZVAJINKPMORJF-UHFFFAOYSA-N


In [10]:
# Remove Duplicate Molecules

df_unique = df.drop_duplicates(subset="InChIKey")

df_unique

,Compound,SMILES,Salt_Removed,Neutralized,Canonical_SMILES,Canonical_Tautomer,InChIKey
0,Ethanol,CCO,CCO,CCO,CCO,CCO,LFQSCWFLJHTTHZ-UHFFFAOYSA-N
1,Acetic Acid,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O,CC(=O)O,QTBSBXVTEAMEQO-UHFFFAOYSA-N
3,Metformin Hydrochloride,CN(C)C(=N)N.Cl,CN(C)C(=N)N,CN(C)C(=N)N,CN(C)C(=N)N,CN(C)C(=N)N,SWSQBOPZIKWTGO-UHFFFAOYSA-N
4,Ammonium,[NH4+],[NH4+],N,N,N,QGZKDVFQNNGYKY-UHFFFAOYSA-N
5,Acetylacetone (Keto),CC(=O)CC(=O)C,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O,CC(=O)CC(C)=O,YRKCREAYFQTBPV-UHFFFAOYSA-N
7,Paracetamol,CC(=O)NC1=CC=C(C=C1)O,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,RZVAJINKPMORJF-UHFFFAOYSA-N
9,Aspirin,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,CC(=O)Oc1ccccc1C(=O)O,BSYNRYMUTXBXSQ-UHFFFAOYSA-N


✓ Molecular standardization is a multi-step preprocessing workflow.

✓ Salts and counter ions are removed to retain the parent molecule.

✓ Charged molecules are neutralized where chemically appropriate.

✓ Canonical SMILES provides a unique textual representation.

✓ Canonical tautomer standardization ensures consistent representation of tautomeric forms.

✓ InChIKeys provide unique molecular identifiers for duplicate detection.

✓ Standardized molecular data is essential for reliable cheminformatics and drug discovery analyses.